---
title: "A Closer Look at una-cybertron-7B-v2 for Summarization"
date: "2023-12-16"
categories: [nlp]
slug: "20231216-una-cybertron-7b-v2-analysis"
description: "There's drama on HuggingFace about data contamination in top-ranked models. I've been using una-cybertron-7B-v2 for summarization—time to take a closer look at what's actually going on."
---



<style>
/* Custom CSS to replicate Bulma styles for select box and text area */

/* Select box */
select.custom-select {
    display: block;
    width: 100%;
    height: calc(2.25em + 2px);
    padding: 0.375em 0.75em;
    font-size: 1em;
    line-height: 1.5;
    color: #495057;
    background-clip: padding-box;
    border: 1px solid #ced4da;
    border-radius: 0.25rem;
    transition: border-color 0.15s ease-in-out, box-shadow 0.15s ease-in-out;
}

/* Text area */
textarea.custom-textarea {
    display: block;
    width: 100%;
    height: calc(2.25em + 280px);
    padding: 0.375em 0.75em;
    font-size: 1em;
    line-height: 1.5;
    color: #495057;
    background-clip: padding-box;
    border: 1px solid #ced4da;
    border-radius: 0.25rem;
    transition: border-color 0.15s ease-in-out, box-shadow 0.15s ease-in-out;
    resize: vertical;
}
</style>

There is some drama going on in the HuggingFace community boards around the possibility of [data contamination](https://bdtechtalks.com/2023/07/17/llm-data-contamination/) on some of the top performing open source LLMs, affecting the reliability of their own [leaderboard](https://huggingface.co/spaces/HuggingFaceH4/open_llm_leaderboard). One of the models under discussion is **una-cybertron-7B-v2**, a Mistral fine-tune that I have been using for summarization tasks with pretty good results. According to the author, the model is tamed using the *Uniform Neural Alignment*, an undisclosed technique on which he might publish a paper later. He also states the following:

>*UNA : Uniform Neural Alignment. It goes in the Attention and the multiperceptrons, and it does what it says. There are multiple phases... Juanako = UNAv1 only implemented at perceptron level. **Cybertron = UNAv2 .. applied both MLP and Attention..** Xaberius = UNAv1.. meaning I can release a much more powerful version of it. Its based on platypus-34b and if u compare the performance.. its not too distant from it. And if u compare what UNA increases (Rationale/Logic capacity).. u'll see the pattern across them.*

A somewhat mysterious and not totally clear message, but interesting nonetheless. The underlying situation is that the community is divided, with some accusing the author of intentionally gaming the leaderboard while others vouch for the superior performance of his models, whatever the underlying fine-tune technique happens to be.

I originally selected the model for my use case (summarization) based on *vibe checks*, as I had the impression that it did produce better summaries compared to the base Mistral or even **OpenHermes-2.5-neural-chat-7B-v3-1-7B** blend, but maybe I was deceived by chance and flashy numbers? In this blog post I aim to sort this out by doing a formal comparison that will hopefully settle things down.

## Summarization Task

To test the models against each other, we will perform a *recursive summarization by parts* task. This involves iteratively summarize a long document into a small set of notes. First, we summarize the paper into notes, then those notes into distilled notes, and further into extra-distilled notes, and so on. We will use the following prompt, applying it iteratively over small chunks of the document until the entire text is processed. Then, we will recursively apply this procedure over the results until we reach a set of summary notes with a total token length of less than 500.

In [4]:
SUMMARIZE_BY_PARTS_TEMPLATE = """You are an applied AI researcher specialized in the field of Large Language Models (LLMs), and you are currently reviewing the academic paper "{paper_title}". Your goal is to analyze the paper, identify the main contributions and most interesting findings, and write a bullet point list summary of it in your own words. This summary will serve as reference for future LLM researchers within your organization, so it is very important that you are able to convey the main ideas in a clear, complete and concise manner.

Read over the following section and take notes. Use a numbered list to summarize the main ideas. 

[...]
{content}
[...]

## Guidelines
- Focus on the bigger picture and the main ideas, rather than on the details. 
- Be sure to explain any new concept or term you introduce. Explain how things work clearly.
- Take notes of the most important numeric results and metrics (i.e. 30% accuracy, 4.5 times faster, etc.).
- If a table is presented just report back the main findings.
- Include examples in your notes that help clarify the main ideas.
- Highlight any practical applications or benefits of the paper's findings.
- Highlight unusual or unexpected findings.
- Take notes in the form of a numbered list. Do not include headers or any other elements.
- Do not include more than 10 items in your list.
- Your summary must be shorter than the original text. Remove any filler or duplicate content.
"""

Finally, to make the output notes easier to digest, we will run them through a final LLM call, applying the following copy-editor prompt for it:

In [5]:
NARRATIVE_SUMMARY_PROMPT = """You are an expert New York Times technology writer tasked with writing a summary of "{paper_title}". Your task is to read the following set of notes and convert them into an engaging paragraph.

{previous_notes}

## Guidelines
- You can reorganize and rephrase the notes in order to improve the summary's flow.
- Do not alter the meaning of the notes.
- Avoid repetition and filler content.
- Abstain from making unwarranted inferences.
- Avoid bombastic language. 
- Include metrics and statistics in your report  (i.e. 30% accuracy, 4.5 times faster, etc.).
- Include descriptions and explanations of any new concepts or terms. Describe how new models or methodologies work.
- Highlight any practical applications or benefits of the paper's findings.
- Highlight unusual or unexpected findings.
"""

## Data
To keep things short, we will focus on summarizing three arXiv papers, the last of which is highly technical. We will manually inspect the results to decide which model performs the best job. The papers under analysis will be the following:

**[Chain-of-Verification Reduces Hallucination in Large Language Models](http://llmpedia.streamlit.app/?arxiv_code=2309.11495)**
>*Abstract:* Generation of plausible yet incorrect factual information, termed hallucination, is an unsolved issue in large language models. We study the ability of language models to deliberate on the responses they give in order to correct their mistakes. We develop the Chain-of-Verification (CoVe) method whereby the model first (i) drafts an initial response; then (ii) plans verification questions to fact-check its draft; (iii) answers those questions independently so the answers are not biased by other responses; and (iv) generates its final verified response. In experiments, we show CoVe decreases hallucinations across a variety of tasks, from list-based questions from Wikidata, closed book MultiSpanQA and longform text generation.

**[Large Language Models Cannot Self-Correct Reasoning Yet](http://llmpedia.streamlit.app/?arxiv_code=2310.01798)**
>*Abstract:* Large Language Models (LLMs) have emerged as a groundbreaking technology with their unparalleled text generation capabilities across various applications. Nevertheless, concerns persist regarding the accuracy and appropriateness of their generated content. A contemporary methodology, self-correction, has been proposed as a remedy to these issues. Building upon this premise, this paper critically examines the role and efficacy of self-correction within LLMs, shedding light on its true potential and limitations. Central to our investigation is the notion of intrinsic self-correction, whereby an LLM attempts to correct its initial responses based solely on its inherent capabilities, without the crutch of external feedback. In the context of reasoning, our research indicates that LLMs struggle to self-correct their responses without external feedback, and at times, their performance might even degrade post self-correction. Drawing from these insights, we offer suggestions for future research and practical applications in this field

**[White-Box Transformers via Sparse Rate Reduction: Compression Is All There Is?](http://llmpedia.streamlit.app/?arxiv_code=2311.13110)**
>*Abstract:* In this paper, we contend that a natural objective of representation learning is to compress and transform the distribution of the data, say sets of tokens, towards a low-dimensional Gaussian mixture supported on incoherent subspaces. The goodness of such a representation can be evaluated by a principled measure, called sparse rate reduction, that simultaneously maximizes the intrinsic information gain and extrinsic sparsity of the learned representation. From this perspective, popular deep network architectures, including transformers, can be viewed as realizing iterative schemes to optimize this measure. Particularly, we derive a transformer block from alternating optimization on parts of this objective: the multi-head self-attention operator compresses the representation by implementing an approximate gradient descent step on the coding rate of the features, and the subsequent multi-layer perceptron sparsifies the features. This leads to a family of white-box transformer-like deep network architectures, named CRATE, which are mathematically fully interpretable. We show, by way of a novel connection between denoising and compression, that the inverse to the aforementioned compressive encoding can be realized by the same class of CRATE architectures. Thus, the so-derived white-box architectures are universal to both encoders and decoders. Experiments show that these networks, despite their simplicity, indeed learn to compress and sparsify representations of large-scale real-world image and text datasets, and achieve performance very close to highly engineered transformer-based models: ViT, MAE, DINO, BERT, and GPT2. We believe the proposed computational framework demonstrates great potential in bridging the gap between theory and practice of deep learning, from a unified perspective of data compression.

## Results

Below we look at the summaries produced by the different models and analyze their quality.

### *Chain-of-Verification Reduces Hallucination in Large Language Models*

<div class="field">
  <div class="control">
    <div class="select is-fullwidth">
      <select id="model_select_a" name="model_select_a" class="custom-select">
        <option value="Mistral-7B-Instruct-v0.2">Mistral-7B-Instruct-v0.2</option>
        <option value="OpenHermes-2.5-neural-chat-7B-v3-1-7B">OpenHermes-2.5-neural-chat-7B-v3-1-7B</option>
        <option value="una-cybertron-7B-v2">una-cybertron-7B-v2</option>
      </select>
    </div>
  </div>
</div>

<div class="field">
  <div class="control">
    <textarea id="summary_text_a" name="summary_text_a" class="custom-textarea"></textarea>
  </div>
</div>

<script>
var model_summaries_a = {
    'Mistral-7B-Instruct-v0.2': "[...] Question Question Question Question,, ,,, Question, Question Question Question, Question, Question Question Question Question Question Question Question Question,,,, Question Question Question Question Question Question Question Question Question Question Question Question Question Question Question Question Question Question Question Question,, Question Question,,,, Question Question Question Question Question Question Question Question Question Question Question,,,,, Question Question Question Question Question Question Question Question Question Question,,,,, Question Question Question,,,,,,, Question Question,,,,,,,,,,,,, in,,, Question Question,,,,,, Question,,, \n Question, Question, Question Question Question Question,,,, a Question Question Question Question Question Question Question Question Question Question Question Question Question Question, ,,,,,,,,,,,,,,,,,,,,,,,,,,,, ,,, , \n,,,,,,,,,,,,,,, Question,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,\n\n\n,,,, , a Question, Question,,,, and, ,, ,,,,,,,, , , ,,,,,\n\n\n\n,,,,,, ,,, Question,,,,,,,,,,,,,,,,,,,,,,,, ,,,,, a Question,,,,, , ,,,,, ,,,,, \n\n\n,,,,,, ,,,,,,,,,,,,,,,,,, , , ,,,,,,,,,,,,,,,,, , a Question,,,, \n\n\n\n,,,, , ,,,, \n\n\n\n,, , ,,,, , ,,,,,,,,,, \n,,,,,, , ,,,,,,, a Question,,,,,,,,,,, , ,,,,,,, Question,,, ,,,,, , ,,,,, ,,,, ,\n, the Question,,,,, , , ,, a Question,,, , ,,, ,,,,,,, , , , \n, Q Question, , , , \n\n\n, \n, ,,,,, ,, , , ,,, , , a Question, , \n\n, ,,,,,, , \n,,, and,,,,,,,,,, , ,,,,, ,,,,,,, ,,, , ,,,,, \n, a Question,,,, , ,,, , Q Question,,,,,,,,,, , , ,, , ,,,, , ,,,, , ,,,,, \n, , ,, , a Question,,,, , , \n\n, , , , , \n, Q, , \n, , , , , , , ,, ,,, and, , , , ,,, , a Question,,,,,, , ,, ,,,,,,,,, ,,, , Q,\n\n\n,,,,, , \n,,,,,, a,,,, , , \n,,,,, and,, , \n,, , \n,,,, ,,,,,, , \n, Q, , , \n, \n\n\n\n,,, , , a Question,,,,,,,,,, , , , \n,,,,,,,, \n\n\n\n\n,,, \n,,, \n, Q, , , \n\n\n\n\n\n\n\n\n,,, , a Question, , and,,,,,, \n,,,,,,,, \n,,,,, \n,, , \n, \n\n\n,,\n, Q,, \n\n\n\n\n\n,,,,,\n,,\n,,, a Question,,,,,,,,,,,,, , , , \n,,,,,, \n\n\n,,,,,,,,,, Q, \n,,,,,,,,,,,,,,,,,,, a,,,,,, \n, \n-\n \n \n,,\n,,,,,,,,,,,,,, in,,,,,,,,,,,,\n,,,,,,,,,,,,,\n \n,,,, Q,,\n\n,,,,,\n, a,,,,,,,,,. I, \n\n,,,,,,,,,,,,,,,,,,,,,,,, \n\n\n\n,\n,,,,,, \n \n\n,,,,,,,,\n\n,,,,,,,,,,, Q,,,, \n\n\n\n \n \n,,,, \n # # # # # # # # # # # # # # # # # # # # # # # # # # # # # \n \n \n \n\n\n,\n  \n \n\n\n \n\n, \n, in a \n\n \n \n \n,,, \n \n \n # \n \n,, Q,,, \n \n,,,,\n,, \n\n \n, \n, \n # # \n\n,,,,, \n \n \n \n a,,\n \n \n \n \n \n \n \n # # # # # # # # # \n\n\n \n # # # # # # # # # # # # # # # # # # -. # # # # # # # # # # # \n # , \n \n # # # # # # # # # # # \n \n \n \n # # # # # # # # # # # # # # \n \n # # # # # # # # \n \n \n # # # # # # , -,\n \n \n # # \n \n \n \n ,  \n # # # # # # # # # # # # # # # \n # # \n # \n \n . # # # # # # # # # # # # # # # \n # # # # # # # # # # # \n,, \n # \n # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # ,,,,,\n # \n \n \n \n \n\n\n\n , a \n # # \n \n \n\n\n \n \n \n # # # # # # # # \n \n # \n \n , \n \n # \n \n \n \n \n \n \n,,,. # # # # # # # # # # # # # # # # # \n \n-,\n \n\n, a \n \n \n \n\n \n,, \n \n # # # \n # # # # \n # \n  \n \n # # # # # # # # # # # # # # # # # # # # # # # # \n \n \n \n \n \n \n . # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # ,,, \n\n-,,,,,,,,,, \n[...]",
    'OpenHermes-2.5-neural-chat-7B-v3-1-7B': "Chain-of-Verification (COVE) is a novel approach to reduce hallucination in large language models by breaking tasks into smaller subtasks and verifying each step before progressing. This method encourages self-critique, leading to improved responses. Practical applications of COVE include AI assistants, chatbots, question-answering systems, healthcare, finance, and AI-assisted writing. While COVE addresses factual inaccuracies, it does not tackle other types of hallucinations like incorrect reasoning or opinions. To improve accuracy and reliability in large language models (LLMs), further research is necessary to address the ongoing challenge of hallucination.\nThe study introduces CoVe Factor, a new method for measuring effectiveness, with an example in longform generation. \nIn experiments, the most effective method was CoVe Factor + Revise, which involved cross-checking executed verifications and regenerating responses using consistent facts. The importance of explicit reasoning and verification techniques in LLMs is emphasized to enhance performance and reduce hallucinations.\nThe paper focuses on reducing hallucination and improving accuracy in LLMs through verification methods like COVE. Further research is encouraged to explore other ways of utilizing verification techniques to enhance LLMs' capabilities. By verifying information consistency, the CoVe method leads to more accurate and trustworthy results in language processing tasks, ultimately benefiting practical applications of these findings and leading to better performance, accuracy, and trustworthiness in large language models.",
    'una-cybertron-7B-v2': "Chain-of-Verification (COVE) is a novel approach to reduce hallucinations in large language models (LLMs). It involves generating an initial response, planning verification questions based on that response, answering those questions independently, and finally incorporating the analysis into a revised response. COVE's performance is comparable to human experts on certain tasks, suggesting its potential for improving AI systems' quality. The method can be applied to any LLM without additional training or fine-tuning. In factored variants, separate verification steps attend only to specific contexts, leading to further performance gains across list-based questions, closed book QA, and longform text generation tasks. Across all three tasks, factored variants consistently outperform the joint approach by 10% in list-based questions, up to 25% in closed book QA, and a 30% increase in accuracy with 4.5 times faster execution time in longform text generation. COVE's verification method can enhance the accuracy and reliability of AI-generated responses in various domains, leading to better user experiences and increased trust in AI systems."
};

// Get the select field and text area elements
var model_select_a = document.getElementById('model_select_a');
var summary_text_a = document.getElementById('summary_text_a');

// Define a function that updates the summary text based on the selected model
function update_summary_text_a() {
    var model_name_a = model_select_a.value;
    summary_text_a.value = model_summaries_a[model_name_a];
}

// Attach the function to the select field
model_select_a.addEventListener('change', update_summary_text_a);

// Call the function once to initialize the text area
update_summary_text_a();
</script>

#### *Analysis*

In this first paper summary, we observe that **Mistral Instruct** forgot to emit the `<|endoftext|>` token, resulting in a never-ending stream of garbage. The other two models perform decently at summarization, although **una-cybertron** does a better job as it adhered to my instruction to reply in a single paragraph, include metrics, and produce a more readable text.
  
| Model | Score |
| --- |------|
| Mistral-7B-Instruct-v0.2 | 0     |
| OpenHermes-2.5-neural-chat-7B-v3-1-7B | 0.7   |
| una-cybertron-7B-v2 | 0.9   |

### *Large Language Models Cannot Self-Correct Reasoning Yet*

<div class="field">
  <div class="control">
    <div class="select is-fullwidth">
      <select id="model_select_b" name="model_select_b" class="custom-select">
        <option value="Mistral-7B-Instruct-v0.2">Mistral-7B-Instruct-v0.2</option>
        <option value="OpenHermes-2.5-neural-chat-7B-v3-1-7B">OpenHermes-2.5-neural-chat-7B-v3-1-7B</option>
        <option value="una-cybertron-7B-v2">una-cybertron-7B-v2</option>
      </select>
    </div>
  </div>
</div>

<div class="field">
  <div class="control">
    <textarea id="summary_text_b" name="summary_text_b" class="custom-textarea"></textarea>
  </div>
</div>

<script>
var model_summaries_b = {
    'Mistral-7B-Instruct-v0.2': "In recent years, Large Language Models (LLMs) have emerged as groundbreaking technology with unparalleled text generation capabilities across various applications. However, concerns persist regarding the accuracy and appropriateness of generated content by LLMs.\nTo address these issues, self-correction has been proposed as a remedy to improve the accuracy and appropriateness of generated content by LLMs. Self-correction refers to the ability of an LLM to refine its responses based on feedback to its previous outputs.\nDespite the potential benefits of self-correction in LLMs, the underlying mechanics and efficacy of self-correction in LLMs remain underexplored. To critically examine the self-correction capabilities of LLMs, a recent paper focused on reasoning as a specific application of self-correction in LLMs.\nThe authors argued that understanding the intrinsic capabilities of LLMs is crucial because high-quality external feedback is unavailable in many real-world applications. The paper reported that when tested on a task involving intrinsic self-correction, an LLM struggled to amend its prior responses and even deteriorated in performance post self-correction.\nThe authors suggested that the limitations of intrinsic self-correction may stem from the fact that LLMs are fundamentally limited by their lack of understanding of the world and the context in which they operate.\nTo improve the ability of LLMs to self-correct reasoning errors effectively, several approaches have been proposed. These include developing more advanced LLM architectures or incorporating external knowledge sources into LLMs.\nOne promising approach is to develop more advanced LLM architectures that are specifically designed to enhance their self-correction capabilities. For example, some researchers have proposed using hierarchical or modular LLM architectures that allow for the separation of different aspects of reasoning and the ability to refine responses based on feedback from multiple sources.\nAnother promising approach is to incorporate external knowledge sources into LLMs in order to enhance their self-correction capabilities by providing them with additional context and information that can help them better understand the world and the context in which they operate. For example, some researchers have proposed using external knowledge sources such as databases, encyclopedias, or other sources of structured or unstructured data to provide LLMs with additional context and information that can help them better understand the world and the context in which they operate.\nIn conclusion, self-correction has been proposed as a remedy to improve the accuracy and appropriateness of generated content by LLMs. However, the underlying mechanics and efficacy of self-correction in LLMs remain underexplored. To critically examine the self-correction capabilities of LLMs, researchers have focused on reasoning as a specific application of self-correction in LLMs. Through their research, these scholars have shed light on the limitations of intrinsic self-correction and the potential benefits of developing more advanced LLM architectures or incorporating external knowledge sources into LLMs to enhance their self-correction capabilities.",
    'OpenHermes-2.5-neural-chat-7B-v3-1-7B': "Large Language Models (LLMs) have shown impressive text generation capabilities, but they struggle with accuracy and appropriateness, making self-correction a potential solution for improving performance. A recent study investigates intrinsic self-correction in LLMs for reasoning tasks without external feedback, revealing that they struggle to self-correct their reasoning effectively and sometimes degrade performance after self-correction. The findings emphasize the need for further exploration into self-correction mechanisms and their impact on LLM performance.\nImproving LLMs' reasoning capabilities is crucial for critical applications in decision-making, education, and healthcare. However, current models like GPT-4 still have limitations in self-correction despite existing methods such as internal and external feedback. External resources significantly impact performance in realistic settings, highlighting the need to improve LLMs' self-correction abilities.\nThe study presents a framework for understanding current limitations and areas of improvement for LLMs, which has significant implications on various applications relying on AI systems' reasoning abilities. Addressing these limitations can lead to more accurate decision-making processes in AI systems, potentially leading to unexpected findings guiding future research efforts in this field. The paper serves as a foundation for advancing the field of LLMs towards self-correction and better reasoning abilities.",
    'una-cybertron-7B-v2': "A recent study has revealed that large language models (LLMs) struggle to self-correct their reasoning without external guidance, indicating a lack of understanding and self-awareness within these AI systems. While LLM performance may improve in some cases, it can also degrade after self-correction, suggesting limited intrinsic self-correction capabilities. This highlights the need for further research into enhancing LLMs' self-correcting abilities. Practical applications could benefit from incorporating human supervision or external feedback to boost model performance. Future work should focus on improving intrinsic self-correction mechanisms and understanding how LLMs learn from their mistakes, addressing safety concerns in AI models, particularly for reasoning tasks. Intrinsic self-correction remains a promising area for research, with potential implications for enhancing the accuracy and reliability of LLMs. The study investigates intrinsic self-correction capabilities without external feedback, crucial for real-world applications. Despite their ability to learn from mistakes and improve through self-correction, LLMs still require additional guidance to achieve perfect accuracy in reasoning tasks. Intrinsic self-correction tests show limited improvements in reasoning benchmarks for GPT-3.5 and GPT-4, indicating that these models struggle with self-correcting their reasoning."
};

// Get the select field and text area elements
var model_select_b = document.getElementById('model_select_b');
var summary_text_b = document.getElementById('summary_text_b');

// Define a function that updates the summary text based on the selected model
function update_summary_text_b() {
    var model_name_b = model_select_b.value;
    summary_text_b.value = model_summaries_b[model_name_b];
}

// Attach the function to the select field
model_select_b.addEventListener('change', update_summary_text_b);

// Call the function once to initialize the text area
update_summary_text_b();
</script>

#### *Analysis*

**Mistral-Instruct** is able to produce a complete and coherent summary for this second paper. Good! Although it's a bit longer than the one paragraph I requested and maintains a bullet-point-like format, the content seems to be of pretty high quality. **OpenHermes-2.5-neural-chat** and **una-cybertron** both do a good job, and I don't think either summary is superior to the other.

| Model | Score |
| --- |------|
| Mistral-7B-Instruct-v0.2 | 0.8  |
| OpenHermes-2.5-neural-chat-7B-v3-1-7B | 0.9  |
| una-cybertron-7B-v2 | 0.9  |

## Results for *White-Box Transformers via Sparse Rate Reduction: Compression Is All There Is?*

<div class="field">
  <div class="control">
    <div class="select is-fullwidth">
      <select id="model_select_c" name="model_select_c" class="custom-select">
        <option value="Mistral-7B-Instruct-v0.2">Mistral-7B-Instruct-v0.2</option>
        <option value="OpenHermes-2.5-neural-chat-7B-v3-1-7B">OpenHermes-2.5-neural-chat-7B-v3-1-7B</option>
        <option value="una-cybertron-7B-v2">una-cybertron-7B-v2</option>
      </select>
    </div>
  </div>
</div>

<div class="field">
  <div class="control">
    <textarea id="summary_text_c" name="summary_text_c" class="custom-textarea"></textarea>
  </div>
</div>

<script>
var model_summaries_c = {
    'Mistral-7B-Instruct-v0.2': "The paper proposes CRATE, a white-box transformer architecture for image classification tasks.\nCRATE is able to learn more efficient and effective representations of data compared to existing deep learning models.\nThe paper demonstrates that CRATE achieves performance very close to these highly engineered transformer-based models on large-scale real-world image and text datasets. [...]\nensancingclensing,densingensensing clensingclensing,clenseancingclensing,<ensing,autoensing,ancesanningersensors thatancingalautoencingersensingersensensing,autoensingensors,\nensive\nensing,\nersensingclensing,autoautoautoautoautoreancingensingclensingensors arecingensingclensingclenses,autoensingingensingauto\ners\n\nensingensinganancesteracingensingensingensors,autoensingsanningensingensingensingensingens\nancingensors.autoatorsencingensors,autoautoautoautoautoersensingensors, autoenseautolingancingensors,fances,clancestersensingensingensingensingensensingensors.cakingsensingancers andautoightensesingensesensingensorsivelensesensingensingensacingensingensingensors,s\nensingensors,ensingensingensors\nensingensors,\nensingensingensingensingensorspaceensingensors;9ensingensors,dantakersensanensesis-\nensors,nancesementensestersancingensors,cight``ancers,auto [...]",
    'OpenHermes-2.5-neural-chat-7B-v3-1-7B': "The 'White-Box Transformers via Sparse Rate Reduction' paper introduces a novel perspective on representation learning as data compression and transformation towards low-dimensional Gaussian mixtures supported on incoherent subspaces. It proposes sparse rate reduction to evaluate this representation, aiming for intrinsic information gain and extrinsic sparsity. The study derives a new transformer block called crate from alternating optimization on parts of the objective, leading to white-box transformer-like architectures with mathematical interpretability. The paper connects denoising and compression, showing that inverse compressive encoding can be realized by the same class of crate architectures. The proposed framework bridges theory and practice in deep learning from a unified perspective of data compression. Networks learn to compress and sparsify representations, achieving performance close to highly engineered transformer-based models. Main contributions include a new perspective on representation learning, the novel measure called sparse rate reduction, and the introduction of white-box transformer-like architectures called crate. Practical applications could be in improving efficiency and interpretability of deep learning models while maintaining performance levels. The paper introduces CRATE, a new approach to representation learning through compressive encoding and decoding for efficient and compact LLMs, achieving significant compression rates without sacrificing performance on various tasks. Future research directions include exploring other methods for evaluating the quality of learned representations in task-agnostic settings and applying the method to other deep learning architectures. The paper explores model compression and optimization through transformers, extending to various neural networks. Deep learning's role in representation learning can improve transformer architectures. Transformers implicitly minimize Kolmogorov complexity related to compression. Improving transformer architectures and their applications require better understanding and optimization. Research on feature representation learning is crucial for enhancing performance. The White-Box Transformers via Sparse Rate Reduction framework aims to improve data distribution learning and sampling efficiency, using a transformer-like encoder for compressing data into structured representations with sparsity and rate reduction measures. The proposed approach outperforms existing methods in accuracy and efficiency on benchmark datasets. A new objective called sparse rate reduction is introduced, optimizing intrinsic complexity (coding rate reduction) and extrinsic simplicity (sparsity). Crate architecture demonstrates competitive performance on various learning tasks using large-scale real-world datasets. Sparse rate reduction leads to linearized, sparse, and compressed representations with low or no redundancy. The main contribution is a practical method for compressing transformer models without sacrificing performance, which can be applied across various domains, including NLP tasks, computer vision, and speech recognition. Efficient token set compression enhances model performance and efficiency while maintaining performance, potentially benefiting resource utilization, faster processing, and more accessible AI models with reduced size. Compression techniques improve model performance and efficiency without compromising performance, opening doors for various applications in different domains.",
    'una-cybertron-7B-v2': "The paper proposes a novel perspective on representation learning, viewing it as compressing and transforming data into low-dimensional Gaussian mixtures supported by incoherent subspaces. This approach introduces sparse rate reduction to evaluate the quality of these representations, maximizing both intrinsic information gain and extrinsic sparsity. The authors argue that popular deep network architectures, including transformers, can be seen as iterative schemes optimizing this measure.\nA white-box transformer-like architecture called CRATE (Compressive Representation Architecture) is derived from alternating optimization on parts of the sparse rate reduction objective. Experiments show that networks learn to compress and sparsify representations in large-scale real-world image and text datasets, achieving performance comparable to highly engineered transformer-based models.\nCRATE (Compressive Rate-based Attention Transformer) is introduced as a white-box compression technique for transformers, aiming for interpretable and efficient models by reducing the number of parameters in these architectures. The paper's main contributions include learning parsimonious representations through unrolled optimization, understanding self-attention as gradient descent on coding rates, and using MLPs for sparse coding.\nThe overall architecture, CRATE (Compressive Rate-based Attention Transformer), is presented with white-box decoding via structured denoising and diffusion. Experimental evaluations demonstrate the effectiveness of CRATE in various practical tasks such as supervised image classification, image completion, self-supervised learning, and pre-training language models.\nCRATE achieves up to 30% parameter reduction while maintaining accuracy, with a speedup of 4.5 times compared to the original model. Sparse Rate Reduction (SRR) is an effective method for compressing large transformer models with minimal loss of performance on downstream tasks, applicable to other deep learning architectures like CNNs and RNNs."
};

// Get the select field and text area elements
var model_select_c = document.getElementById('model_select_c');
var summary_text_c = document.getElementById('summary_text_c');

// Define a function that updates the summary text based on the selected model
function update_summary_text_c() {
    var model_name_c = model_select_c.value;
    summary_text_c.value = model_summaries_c[model_name_c];
}

// Attach the function to the select field
model_select_c.addEventListener('change', update_summary_text_c);

// Call the function once to initialize the text area
update_summary_text_c();
</script>

#### *Analysis*
For the third paper, we again observe that **Mistral Instruct** fails to emit the termination token and produces a senseless stream of text. Both **OpenHermes-2.5-neural-chat** and **una-cyberton** do a decent job, although the latter model earns additional points for including numerical metrics, a requested, and for producing a summary that better captures the bigger picture and is more enjoyable to read.

| Model | Score |
| --- |-------|
| Mistral-7B-Instruct-v0.2 | 0     |
| OpenHermes-2.5-neural-chat-7B-v3-1-7B | 0.7   |
| una-cybertron-7B-v2 | 0.9   |

## Concluding Remarks

Looking at aggregate results it does seem like there is something noteworthy about the **una-cybertron** model. It appears to be at least on par with the state-of-the-art **OpenHermes-2.5-neural-chat-7B-v3-1-7B**, and possibly even follows instructions slightly better.

| Model | Score |
| --- |-------|
| Mistral-7B-Instruct-v0.2 | 0.27  |
| OpenHermes-2.5-neural-chat-7B-v3-1-7B | 0.76  |
| una-cybertron-7B-v2 | 0.9   |
